# Cattle breed classification on Colab

Anyone on the team can use this notebook as it is. You don't need to edit any code.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.

- **Code** comes from GitHub. Push your work first, then type your branch name in the Setup cell (`main` if you work on main).
- **Datasets and trained models** are kept in *your* Google Drive under **My Drive → cattle-breed**. They survive disconnects and are downloaded only once per person.
- **Using a teammate's models:** they share their `cattle-breed` Drive folder with you. In Drive, choose *Organize → Add shortcut → My Drive*, and keep the name `cattle-breed`. Only one person should train at a time in a shared folder.
- **After a disconnect:** run Setup again and continue where you stopped. Training resumes on its own.
- **Don't commit this notebook with outputs.** Choose *Edit → Clear all outputs* first, so teammates don't get merge conflicts.

## Setup (run every time)

In [ ]:
BRANCH = "main"  # @param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")

%cd /content
!test -d CattleClass || git clone -q https://github.com/ShayanAhmed12/CattleClass.git
%cd /content/CattleClass
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH} && git log --oneline -1
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU! Runtime > Change runtime type > T4 GPU"
!python -m pytest -q

## Step 1: data
Prepare CID first so model training can begin while the external images are reviewed privately.

In [ ]:
!python prepare_data.py --datasets cid

## Step 2: training (B0, E1, E2, final model)
If Colab disconnects, run Setup again and then this cell. Folds that are already finished are skipped.

In [ ]:
!python train.py

In [ ]:
!python train.py --final

## Step 3: evaluation (E3, E4, E5, figures)
Download the external datasets with `python prepare_data.py --datasets mendeley kaggle`, then follow the private T3 review in `docs/data.md`.
Run the cell below after review. It rebuilds the caches with your exclusions and runs the official evaluation.

In [ ]:
!python prepare_data.py --datasets mendeley kaggle && python evaluate.py --robustness --figures

## Step 4: demo
This prints a public link you can open on any device.

In [ ]:
!python demo.py --share

## Save the results

`results/` (metrics and figures) lives in this temporary Colab copy, so save it before you close the tab.

**Option A:** copy the metrics and aggregate plots to your Drive. The CID sample grid and the local error analysis folder stay in the current runtime.

In [ ]:
from shutil import copytree, ignore_patterns

copytree("results", "/content/drive/MyDrive/cattle-breed/results", dirs_exist_ok=True,
         ignore=ignore_patterns("cid_samples.png", "local"))

**Option B:** commit and push straight to your branch on GitHub. You need write access to the repository.

One-time setup:
1. On GitHub, open *Settings → Developer settings → Fine-grained tokens* and create a token with *Contents: read and write* for CattleClass.
2. In Colab, open the key icon on the left and add three secrets:
   - `GH_TOKEN`: the token;
   - `GIT_NAME`: your name as it should appear on commits;
   - `GIT_EMAIL`: your email.

The commit is made under your own name. Teammates' newer commits are pulled in first.

In [ ]:
import subprocess
from google.colab import userdata

def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True)

try:
    token = userdata.get("GH_TOKEN")
    git("config", "user.name", userdata.get("GIT_NAME"))
    git("config", "user.email", userdata.get("GIT_EMAIL"))
except Exception:
    raise SystemExit("Add the GH_TOKEN, GIT_NAME and GIT_EMAIL secrets first (key icon on the left).")

url = f"https://{token}@github.com/ShayanAhmed12/CattleClass.git"
git("add", "results")
git("commit", "-q", "-m", "Results from Colab")
pulled = git("pull", "-q", "--rebase", url, BRANCH)
pushed = git("push", "-q", url, f"HEAD:{BRANCH}")
if pushed.returncode == 0:
    print(f"Pushed to {BRANCH}.")
else:
    print("Push failed; use Option A instead.")
    print((pulled.stderr + pushed.stderr).replace(token, "***"))